In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import gc
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cv2
import nibabel as nib

import tensorflow as tf

from scipy.ndimage import binary_fill_holes
from scipy.spatial.distance import cdist


print("TensorFlow:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPU(s):", gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(
                gpu,
                True
            )
        print("✓ GPU memory growth enabled")

    except RuntimeError as e:
        print("GPU memory-growth warning:", e)

TensorFlow: 2.20.0
GPU(s): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✓ GPU memory growth enabled


In [ ]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("✓ TensorFlow deterministic operations enabled")
except Exception as e:
    print("Determinism note:", e)

print("Seed:", SEED)

✓ TensorFlow deterministic operations enabled
Seed: 42


In [ ]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/CAMUS_FINAL"
)

PHASE1_DIR = (
    PROJECT_DIR /
    "phase1_segmentation"
)

P1A_DIR = (
    PHASE1_DIR /
    "phase1A_resolution"
)

P1B_DIR = (
    PHASE1_DIR /
    "phase1B_hyperparameters"
)

P1B_CONFIG_DIR = (
    P1B_DIR /
    "configs"
)

P1B_MODEL_DIR = (
    P1B_DIR /
    "models"
)

P1B_HISTORY_DIR = (
    P1B_DIR /
    "histories"
)

P1B_TABLE_DIR = (
    P1B_DIR /
    "tables"
)

P1B_PLOT_DIR = (
    P1B_DIR /
    "plots"
)


for folder in [
    P1B_CONFIG_DIR,
    P1B_MODEL_DIR,
    P1B_HISTORY_DIR,
    P1B_TABLE_DIR,
    P1B_PLOT_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


print("Project:")
print(PROJECT_DIR)

print("\nPhase 1B:")
print(P1B_DIR)

print("\n✓ Phase 1B paths ready.")

Project:
/content/drive/MyDrive/CAMUS_FINAL

Phase 1B:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters

✓ Phase 1B paths ready.


In [ ]:
# ------------------------------------------------
# Phase 1A resolution decision
# ------------------------------------------------

RESOLUTION_DECISION_PATH = (
    P1A_DIR /
    "configs" /
    "phase1A_final_resolution_decision.json"
)

assert RESOLUTION_DECISION_PATH.exists(), (
    "Phase 1A resolution decision not found."
)

with open(
    RESOLUTION_DECISION_PATH,
    "r"
) as f:
    resolution_decision = json.load(f)


SELECTED_RESOLUTION = int(
    resolution_decision[
        "selected_resolution"
    ]
)


# ------------------------------------------------
# Phase 1B-1 learning-rate decision
# ------------------------------------------------

LR_DECISION_PATH = (
    P1B_CONFIG_DIR /
    "phase1B1_learning_rate_decision.json"
)

assert LR_DECISION_PATH.exists(), (
    "Phase 1B-1 learning-rate decision not found."
)

with open(
    LR_DECISION_PATH,
    "r"
) as f:
    lr_decision = json.load(f)


SELECTED_LEARNING_RATE = float(
    lr_decision[
        "selected_learning_rate"
    ]
)


# ------------------------------------------------
# Verification
# ------------------------------------------------

assert SELECTED_RESOLUTION == 256

assert np.isclose(
    SELECTED_LEARNING_RATE,
    1e-4
)


print(
    "Locked resolution:",
    f"{SELECTED_RESOLUTION}×{SELECTED_RESOLUTION}"
)

print(
    "Locked learning rate:",
    SELECTED_LEARNING_RATE
)

print("\n✓ Previous experimental decisions loaded.")

Locked resolution: 256×256
Locked learning rate: 0.0001

✓ Previous experimental decisions loaded.


In [ ]:
DATASET_PATH = Path(
    "/content/camus_data_final/database_nifti"
)

if DATASET_PATH.exists():

    print("✓ CAMUS dataset already extracted.")
    print(DATASET_PATH)

else:

    print(
        "CAMUS dataset is not currently "
        "available in this runtime."
    )

CAMUS dataset is not currently available in this runtime.


In [ ]:
import zipfile

OUTER_ZIP_PATH = Path(
    "/content/drive/MyDrive/download.zip"
)

OUTER_EXTRACT_DIR = Path(
    "/content/camus_data"
)

INNER_ARCHIVE_PATH = (
    OUTER_EXTRACT_DIR /
    "download"
)

INNER_EXTRACT_DIR = Path(
    "/content/camus_data_final"
)

DATASET_PATH = (
    INNER_EXTRACT_DIR /
    "database_nifti"
)


if not DATASET_PATH.exists():

    assert OUTER_ZIP_PATH.exists(), (
        f"Archive not found: {OUTER_ZIP_PATH}"
    )

    if not INNER_ARCHIVE_PATH.exists():

        print("Extracting outer archive...")

        OUTER_EXTRACT_DIR.mkdir(
            parents=True,
            exist_ok=True
        )

        with zipfile.ZipFile(
            OUTER_ZIP_PATH,
            "r"
        ) as archive:

            archive.extractall(
                OUTER_EXTRACT_DIR
            )

        print("✓ Outer archive extracted.")


    print("Extracting CAMUS archive...")

    INNER_EXTRACT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(
        INNER_ARCHIVE_PATH,
        "r"
    ) as archive:

        archive.extractall(
            INNER_EXTRACT_DIR
        )


assert DATASET_PATH.exists()

print("\n✓ CAMUS dataset ready:")
print(DATASET_PATH)

Extracting outer archive...
✓ Outer archive extracted.
Extracting CAMUS archive...

✓ CAMUS dataset ready:
/content/camus_data_final/database_nifti


In [ ]:
SPLIT_PATH = (
    PROJECT_DIR /
    "splits" /
    "patient_split_seed42.json"
)

assert SPLIT_PATH.exists(), (
    f"Split file not found:\n{SPLIT_PATH}"
)


with open(
    SPLIT_PATH,
    "r"
) as f:
    patient_split = json.load(f)


train_ids = patient_split.get(
    "train_ids",
    patient_split.get("train")
)

val_ids = patient_split.get(
    "val_ids",
    patient_split.get(
        "validation",
        patient_split.get("val")
    )
)

test_ids = patient_split.get(
    "test_ids",
    patient_split.get("test")
)


assert train_ids is not None
assert val_ids is not None
assert test_ids is not None

assert len(train_ids) == 400
assert len(val_ids) == 50
assert len(test_ids) == 50

assert not (
    set(train_ids) &
    set(val_ids)
)

assert not (
    set(train_ids) &
    set(test_ids)
)

assert not (
    set(val_ids) &
    set(test_ids)
)


print("Train:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

print("\n✓ Original patient-level split verified.")
print("✓ Test set remains excluded from model selection.")

Train: 400
Validation: 50
Test: 50

✓ Original patient-level split verified.
✓ Test set remains excluded from model selection.


In [ ]:
BASELINE_PARAMETER_COUNT = 1_940_817
BATCH_SIZE = 4
MAX_EPOCHS = 30
THRESHOLD = 0.50


patient_folders = sorted([
    p
    for p in DATASET_PATH.iterdir()
    if p.is_dir()
    and p.name.startswith("patient")
])


assert len(patient_folders) == 500


print("=" * 70)
print("PHASE 1B-2 — OPTIMIZER EXPERIMENT")
print("=" * 70)

print("\nLOCKED FROM PREVIOUS EXPERIMENTS")

print(
    "Resolution        :",
    f"{SELECTED_RESOLUTION}×{SELECTED_RESOLUTION}"
)

print(
    "Learning rate     :",
    SELECTED_LEARNING_RATE
)

print(
    "Architecture      : Baseline U-Net"
)

print(
    "Parameters        :",
    BASELINE_PARAMETER_COUNT
)

print(
    "Batch size        :",
    BATCH_SIZE
)

print(
    "Loss              : BCE + Dice"
)

print(
    "Augmentation      : None"
)

print(
    "Epoch budget      :",
    MAX_EPOCHS
)

print(
    "Threshold         :",
    THRESHOLD
)


print("\nVARIABLE BEING TESTED")
print("Optimizer")

print("\nOptimizers:")
print("1. Adam")
print("2. AdamW")

print("\nDataset:")
print("Train patients     :", len(train_ids))
print("Validation patients:", len(val_ids))
print("Test patients      :", len(test_ids))

print("\nTest used for optimizer selection: NO")

print("\n✓ Phase 1B-2 setup verified.")
print("✓ Ready to build the controlled optimizer experiment.")

PHASE 1B-2 — OPTIMIZER EXPERIMENT

LOCKED FROM PREVIOUS EXPERIMENTS
Resolution        : 256×256
Learning rate     : 0.0001
Architecture      : Baseline U-Net
Parameters        : 1940817
Batch size        : 4
Loss              : BCE + Dice
Augmentation      : None
Epoch budget      : 30
Threshold         : 0.5

VARIABLE BEING TESTED
Optimizer

Optimizers:
1. Adam
2. AdamW

Dataset:
Train patients     : 400
Validation patients: 50
Test patients      : 50

Test used for optimizer selection: NO

✓ Phase 1B-2 setup verified.
✓ Ready to build the controlled optimizer experiment.


In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 9
# CAMUS LOADING + PREPROCESSING
# ================================================================

def load_nifti_2d(file_path):

    nii = nib.load(str(file_path))
    data = np.asarray(
        nii.get_fdata(),
        dtype=np.float32
    )

    if data.ndim == 3 and data.shape[-1] == 1:
        data = data[:, :, 0]

    if data.ndim != 2:
        raise ValueError(
            f"Expected 2D image, got {data.shape}"
        )

    return data


def load_camus_frame(patient_id, phase):

    patient_dir = DATASET_PATH / patient_id

    image = load_nifti_2d(
        patient_dir /
        f"{patient_id}_4CH_{phase}.nii.gz"
    )

    full_mask = load_nifti_2d(
        patient_dir /
        f"{patient_id}_4CH_{phase}_gt.nii.gz"
    )

    # CAMUS label 1 = LV cavity
    lv_mask = (
        full_mask == 1
    ).astype(np.float32)

    return image, lv_mask


def preprocess_frame(
    image,
    mask,
    resolution=SELECTED_RESOLUTION
):

    image_resized = cv2.resize(
        image,
        (resolution, resolution),
        interpolation=cv2.INTER_LINEAR
    )

    mask_resized = cv2.resize(
        mask,
        (resolution, resolution),
        interpolation=cv2.INTER_NEAREST
    )

    lo = float(image_resized.min())
    hi = float(image_resized.max())

    if hi > lo:

        image_resized = (
            image_resized - lo
        ) / (hi - lo)

    else:

        image_resized = np.zeros_like(
            image_resized,
            dtype=np.float32
        )

    image_resized = image_resized.astype(
        np.float32
    )

    mask_resized = (
        mask_resized > 0.5
    ).astype(np.float32)

    return (
        image_resized[..., None],
        mask_resized[..., None]
    )


print("✓ Data loading and preprocessing ready.")

✓ Data loading and preprocessing ready.


In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 10
# MEMORY-FRIENDLY DATASETS
# ================================================================

def frame_generator(
    patient_ids,
    shuffle=True,
    seed=SEED
):

    frame_list = [
        (pid, phase)
        for pid in patient_ids
        for phase in ["ED", "ES"]
    ]

    rng = np.random.default_rng(seed)

    while True:

        indices = np.arange(
            len(frame_list)
        )

        if shuffle:
            rng.shuffle(indices)

        for index in indices:

            pid, phase = frame_list[index]

            image, mask = load_camus_frame(
                pid,
                phase
            )

            x, y = preprocess_frame(
                image,
                mask
            )

            yield x, y


TRAIN_FRAMES = len(train_ids) * 2
VAL_FRAMES = len(val_ids) * 2

TRAIN_STEPS = (
    TRAIN_FRAMES // BATCH_SIZE
)

VAL_STEPS = (
    VAL_FRAMES // BATCH_SIZE
)


output_signature = (

    tf.TensorSpec(
        shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        ),
        dtype=tf.float32
    ),

    tf.TensorSpec(
        shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        ),
        dtype=tf.float32
    )
)


train_dataset = (
    tf.data.Dataset
    .from_generator(
        lambda: frame_generator(
            train_ids,
            shuffle=True,
            seed=SEED
        ),
        output_signature=output_signature
    )
    .batch(
        BATCH_SIZE,
        drop_remainder=True
    )
    .prefetch(1)
)


val_dataset = (
    tf.data.Dataset
    .from_generator(
        lambda: frame_generator(
            val_ids,
            shuffle=False,
            seed=SEED
        ),
        output_signature=output_signature
    )
    .batch(
        BATCH_SIZE,
        drop_remainder=False
    )
    .prefetch(1)
)


print("Training frames :", TRAIN_FRAMES)
print("Validation frames:", VAL_FRAMES)
print("Training steps  :", TRAIN_STEPS)
print("Validation steps:", VAL_STEPS)

print("\n✓ Datasets ready.")

Training frames : 800
Validation frames: 100
Training steps  : 200
Validation steps: 25

✓ Datasets ready.


In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 11
# LOSS + METRICS
# ================================================================

SMOOTH = 1e-6


def dice_coefficient(y_true, y_pred):

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred,
        tf.float32
    )

    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3]
    )

    denominator = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3]
        )
    )

    dice = (
        2.0 * intersection + SMOOTH
    ) / (
        denominator + SMOOTH
    )

    return tf.reduce_mean(dice)


def iou_metric(y_true, y_pred):

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred,
        tf.float32
    )

    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3]
    )

    union = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3]
        )
        -
        intersection
    )

    iou = (
        intersection + SMOOTH
    ) / (
        union + SMOOTH
    )

    return tf.reduce_mean(iou)


def dice_loss(y_true, y_pred):

    return (
        1.0 -
        dice_coefficient(
            y_true,
            y_pred
        )
    )


def bce_dice_loss(y_true, y_pred):

    bce = (
        tf.keras.losses
        .binary_crossentropy(
            y_true,
            y_pred
        )
    )

    bce = tf.reduce_mean(bce)

    return (
        bce +
        dice_loss(
            y_true,
            y_pred
        )
    )


print("✓ BCE + Dice loss ready.")
print("✓ Dice and IoU ready.")

✓ BCE + Dice loss ready.
✓ Dice and IoU ready.


In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 12
# FIXED BASELINE U-NET
# ================================================================

def conv_block(x, filters):

    x = tf.keras.layers.Conv2D(
        filters,
        3,
        activation="relu",
        padding="same"
    )(x)

    x = tf.keras.layers.Conv2D(
        filters,
        3,
        activation="relu",
        padding="same"
    )(x)

    return x


def build_baseline_unet(
    input_shape=(256, 256, 1)
):

    inputs = tf.keras.Input(
        shape=input_shape
    )

    c1 = conv_block(inputs, 16)
    p1 = tf.keras.layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 32)
    p2 = tf.keras.layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 64)
    p3 = tf.keras.layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 128)
    p4 = tf.keras.layers.MaxPooling2D()(c4)

    c5 = conv_block(p4, 256)

    u6 = tf.keras.layers.Conv2DTranspose(
        128,
        2,
        strides=2,
        padding="same"
    )(c5)

    u6 = tf.keras.layers.Concatenate()(
        [u6, c4]
    )

    c6 = conv_block(u6, 128)

    u7 = tf.keras.layers.Conv2DTranspose(
        64,
        2,
        strides=2,
        padding="same"
    )(c6)

    u7 = tf.keras.layers.Concatenate()(
        [u7, c3]
    )

    c7 = conv_block(u7, 64)

    u8 = tf.keras.layers.Conv2DTranspose(
        32,
        2,
        strides=2,
        padding="same"
    )(c7)

    u8 = tf.keras.layers.Concatenate()(
        [u8, c2]
    )

    c8 = conv_block(u8, 32)

    u9 = tf.keras.layers.Conv2DTranspose(
        16,
        2,
        strides=2,
        padding="same"
    )(c8)

    u9 = tf.keras.layers.Concatenate()(
        [u9, c1]
    )

    c9 = conv_block(u9, 16)

    outputs = tf.keras.layers.Conv2D(
        1,
        1,
        activation="sigmoid"
    )(c9)

    segmentation_model = tf.keras.Model(
        inputs,
        outputs,
        name="Baseline_UNet"
    )

    return segmentation_model

In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 13
# VERIFY ARCHITECTURE
# ================================================================

tf.keras.backend.clear_session()
gc.collect()

architecture_check_model = (
    build_baseline_unet(
        input_shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        )
    )
)

actual_parameters = (
    architecture_check_model.count_params()
)

print(
    "Parameter count:",
    actual_parameters
)

assert (
    actual_parameters ==
    BASELINE_PARAMETER_COUNT
), (
    "Architecture mismatch!"
)

del architecture_check_model

tf.keras.backend.clear_session()
gc.collect()

print("✓ Architecture verified.")

Parameter count: 1940817
✓ Architecture verified.


In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 14
# COMMON INITIALIZATION
# ================================================================

OPT_INITIAL_WEIGHTS_PATH = (
    P1B_CONFIG_DIR /
    "optimizer_experiment_initial_weights.weights.h5"
)


tf.keras.backend.clear_session()
gc.collect()

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


initialization_model = (
    build_baseline_unet(
        input_shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        )
    )
)


initialization_model.save_weights(
    OPT_INITIAL_WEIGHTS_PATH
)


print("Common initialization saved:")
print(OPT_INITIAL_WEIGHTS_PATH)

print(
    "\nParameters:",
    initialization_model.count_params()
)


del initialization_model

tf.keras.backend.clear_session()
gc.collect()

print(
    "\n✓ Adam and AdamW will start "
    "from identical weights."
)

Common initialization saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/configs/optimizer_experiment_initial_weights.weights.h5

Parameters: 1940817

✓ Adam and AdamW will start from identical weights.


In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 15
# FREEZE OPTIMIZER EXPERIMENT PROTOCOL
# ================================================================

OPTIMIZERS_TO_TEST = [
    "Adam",
    "AdamW"
]

ADAMW_WEIGHT_DECAY = 1e-4


optimizer_protocol = {

    "experiment":
        "Phase 1B-2 — Controlled Optimizer Experiment",

    "objective":
        "Compare Adam and AdamW while holding all other "
        "experimental settings fixed.",

    "resolution":
        SELECTED_RESOLUTION,

    "learning_rate":
        SELECTED_LEARNING_RATE,

    "architecture":
        "Baseline U-Net",

    "parameters":
        BASELINE_PARAMETER_COUNT,

    "optimizers":
        OPTIMIZERS_TO_TEST,

    "Adam_configuration": {
        "learning_rate":
            SELECTED_LEARNING_RATE
    },

    "AdamW_configuration": {
        "learning_rate":
            SELECTED_LEARNING_RATE,
        "weight_decay":
            ADAMW_WEIGHT_DECAY
    },

    "batch_size":
        BATCH_SIZE,

    "loss":
        "BCE + Dice",

    "augmentation":
        False,

    "epochs":
        MAX_EPOCHS,

    "checkpoint_monitor":
        "val_dice_coefficient",

    "checkpoint_mode":
        "max",

    "threshold":
        THRESHOLD,

    "seed":
        SEED,

    "same_initial_weights":
        True,

    "initial_weights_path":
        str(
            OPT_INITIAL_WEIGHTS_PATH
        ),

    "train_patients":
        len(train_ids),

    "validation_patients":
        len(val_ids),

    "test_used":
        False,

    "important_note":
        (
            "AdamW weight decay was fixed at 1e-4 "
            "before optimizer training. Weight decay "
            "is not tuned in this experiment."
        )
}


OPT_PROTOCOL_PATH = (
    P1B_CONFIG_DIR /
    "optimizer_experiment_protocol.json"
)


with open(
    OPT_PROTOCOL_PATH,
    "w"
) as f:

    json.dump(
        optimizer_protocol,
        f,
        indent=2
    )


print("OPTIMIZER PROTOCOL FROZEN")
print("-" * 50)

print("Adam:")
print(
    " LR =",
    SELECTED_LEARNING_RATE
)

print("\nAdamW:")
print(
    " LR =",
    SELECTED_LEARNING_RATE
)

print(
    " Weight decay =",
    ADAMW_WEIGHT_DECAY
)

print("\nProtocol:")
print(OPT_PROTOCOL_PATH)

print(
    "\n✓ No optimizer settings should "
    "be changed after this point."
)

OPTIMIZER PROTOCOL FROZEN
--------------------------------------------------
Adam:
 LR = 0.0001

AdamW:
 LR = 0.0001
 Weight decay = 0.0001

Protocol:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/configs/optimizer_experiment_protocol.json

✓ No optimizer settings should be changed after this point.


In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 16
# VERIFY OPTIMIZERS
# ================================================================

adam_check = tf.keras.optimizers.Adam(
    learning_rate=SELECTED_LEARNING_RATE
)

adamw_check = tf.keras.optimizers.AdamW(
    learning_rate=SELECTED_LEARNING_RATE,
    weight_decay=ADAMW_WEIGHT_DECAY
)


print("Adam:")
print(adam_check.get_config())

print("\nAdamW:")
print(adamw_check.get_config())

print("\n✓ Both optimizers available.")

del adam_check
del adamw_check

Adam:
{'name': 'adam', 'learning_rate': 9.999999747378752e-05, 'weight_decay': None, 'clipnorm': None, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': False, 'ema_momentum': 0.99, 'ema_overwrite_frequency': None, 'loss_scale_factor': None, 'gradient_accumulation_steps': None, 'beta_1': 0.9, 'beta_2': 0.999, 'epsilon': 1e-07, 'amsgrad': False}

AdamW:
{'name': 'adamw', 'learning_rate': 9.999999747378752e-05, 'weight_decay': 0.0001, 'clipnorm': None, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': False, 'ema_momentum': 0.99, 'ema_overwrite_frequency': None, 'loss_scale_factor': None, 'gradient_accumulation_steps': None, 'beta_1': 0.9, 'beta_2': 0.999, 'epsilon': 1e-07, 'amsgrad': False}

✓ Both optimizers available.


In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 17
# OPTIMIZER TRAINING FUNCTION
# ================================================================

def build_optimizer(
    optimizer_name
):

    if optimizer_name == "Adam":

        return tf.keras.optimizers.Adam(
            learning_rate=
                SELECTED_LEARNING_RATE
        )

    elif optimizer_name == "AdamW":

        return tf.keras.optimizers.AdamW(
            learning_rate=
                SELECTED_LEARNING_RATE,

            weight_decay=
                ADAMW_WEIGHT_DECAY
        )

    else:

        raise ValueError(
            f"Unknown optimizer: {optimizer_name}"
        )


def train_optimizer_experiment(
    optimizer_name
):

    run_name = optimizer_name.lower()

    print("\n" + "=" * 70)
    print(
        f"TRAINING OPTIMIZER = {optimizer_name}"
    )
    print("=" * 70)

    # Clean memory
    tf.keras.backend.clear_session()
    gc.collect()

    # Reset seeds
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    # Fresh model
    segmentation_model = (
        build_baseline_unet(
            input_shape=(
                SELECTED_RESOLUTION,
                SELECTED_RESOLUTION,
                1
            )
        )
    )

    # IDENTICAL initialization
    segmentation_model.load_weights(
        OPT_INITIAL_WEIGHTS_PATH
    )

    optimizer_instance = (
        build_optimizer(
            optimizer_name
        )
    )

    segmentation_model.compile(

        optimizer=optimizer_instance,

        loss=bce_dice_loss,

        metrics=[
            dice_coefficient,
            iou_metric
        ]
    )

    model_path = (
        P1B_MODEL_DIR /
        f"optimizer_{run_name}_best.keras"
    )

    history_path = (
        P1B_HISTORY_DIR /
        f"optimizer_{run_name}_history.csv"
    )

    checkpoint = (
        tf.keras.callbacks.ModelCheckpoint(

            filepath=str(model_path),

            monitor=
                "val_dice_coefficient",

            mode="max",

            save_best_only=True,

            verbose=1
        )
    )

    start_time = time.time()

    history = segmentation_model.fit(

        train_dataset,

        validation_data=val_dataset,

        epochs=MAX_EPOCHS,

        steps_per_epoch=TRAIN_STEPS,

        validation_steps=VAL_STEPS,

        callbacks=[
            checkpoint
        ],

        verbose=1
    )

    training_minutes = (
        time.time() - start_time
    ) / 60.0

    # Save complete history
    history_df = pd.DataFrame(
        history.history
    )

    history_df.index = (
        np.arange(
            len(history_df)
        ) + 1
    )

    history_df.index.name = "epoch"

    history_df.to_csv(
        history_path
    )

    # Best epoch
    val_dice_values = np.asarray(
        history.history[
            "val_dice_coefficient"
        ],
        dtype=float
    )

    best_index = int(
        np.argmax(
            val_dice_values
        )
    )

    best_epoch = (
        best_index + 1
    )

    result = {

        "optimizer":
            optimizer_name,

        "learning_rate":
            SELECTED_LEARNING_RATE,

        "weight_decay":
            (
                ADAMW_WEIGHT_DECAY
                if optimizer_name == "AdamW"
                else 0.0
            ),

        "parameters":
            int(
                segmentation_model.count_params()
            ),

        "best_epoch":
            best_epoch,

        "best_val_dice_training":
            float(
                val_dice_values[
                    best_index
                ]
            ),

        "val_iou_at_best_dice_epoch":
            float(
                history.history[
                    "val_iou_metric"
                ][best_index]
            ),

        "training_minutes":
            float(
                training_minutes
            ),

        "model_path":
            str(model_path),

        "history_path":
            str(history_path)
    }

    print("\n" + "-" * 70)
    print(
        f"{optimizer_name} COMPLETE"
    )

    print(
        "Best epoch:",
        result["best_epoch"]
    )

    print(
        "Best validation Dice:",
        round(
            result[
                "best_val_dice_training"
            ],
            6
        )
    )

    print(
        "Validation IoU:",
        round(
            result[
                "val_iou_at_best_dice_epoch"
            ],
            6
        )
    )

    print(
        "Training time:",
        round(
            training_minutes,
            2
        ),
        "minutes"
    )

    print(
        "Checkpoint:",
        model_path
    )

    print("-" * 70)

    del segmentation_model
    del optimizer_instance
    del history

    tf.keras.backend.clear_session()
    gc.collect()

    return result

In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 18
# RUN 1 — ADAM
# ================================================================

adam_result = (
    train_optimizer_experiment(
        "Adam"
    )
)

adam_result


TRAINING OPTIMIZER = Adam
Epoch 1/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - dice_coefficient: 0.1832 - iou_metric: 0.1053 - loss: 1.3305
Epoch 1: val_dice_coefficient improved from None to 0.43977, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/optimizer_adam_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/optimizer_adam_best.keras
200/200 ━━━━━━━━━━━━━━━━━━━━ 33s 108ms/step - dice_coefficient: 0.2759 - iou_metric: 0.1687 - loss: 1.0598 - val_dice_coefficient: 0.4398 - val_iou_metric: 0.2837 - val_loss: 0.7019
Epoch 2/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - dice_coefficient: 0.5634 - iou_metric: 0.4030 - loss: 0.6001
Epoch 2: val_dice_coefficient improved from 0.43977 to 0.72617, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/optimizer_adam_best.keras

Epoch 2: finished saving model

{'optimizer': 'Adam',
 'learning_rate': 0.0001,
 'weight_decay': 0.0,
 'parameters': 1940817,
 'best_epoch': 30,
 'best_val_dice_training': 0.9254956245422363,
 'val_iou_at_best_dice_epoch': 0.8635912537574768,
 'training_minutes': 10.544690990447998,
 'model_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/optimizer_adam_best.keras',
 'history_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/histories/optimizer_adam_history.csv'}

In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 19
# RUN 2 — ADAMW
# ================================================================

adamw_result = (
    train_optimizer_experiment(
        "AdamW"
    )
)

adamw_result


TRAINING OPTIMIZER = AdamW
Epoch 1/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - dice_coefficient: 0.1832 - iou_metric: 0.1053 - loss: 1.3305
Epoch 1: val_dice_coefficient improved from None to 0.43977, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/optimizer_adamw_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/optimizer_adamw_best.keras
200/200 ━━━━━━━━━━━━━━━━━━━━ 27s 100ms/step - dice_coefficient: 0.2759 - iou_metric: 0.1687 - loss: 1.0598 - val_dice_coefficient: 0.4398 - val_iou_metric: 0.2837 - val_loss: 0.7019
Epoch 2/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - dice_coefficient: 0.5634 - iou_metric: 0.4030 - loss: 0.6001
Epoch 2: val_dice_coefficient improved from 0.43977 to 0.72617, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/optimizer_adamw_best.keras

Epoch 2: finished saving m

{'optimizer': 'AdamW',
 'learning_rate': 0.0001,
 'weight_decay': 0.0001,
 'parameters': 1940817,
 'best_epoch': 30,
 'best_val_dice_training': 0.9254956245422363,
 'val_iou_at_best_dice_epoch': 0.8635912537574768,
 'training_minutes': 10.582544549306233,
 'model_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/optimizer_adamw_best.keras',
 'history_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/histories/optimizer_adamw_history.csv'}

In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 20
# TRAINING COMPARISON
# ================================================================

optimizer_training_summary = (
    pd.DataFrame([
        adam_result,
        adamw_result
    ])
)


OPT_TRAINING_SUMMARY_PATH = (
    P1B_TABLE_DIR /
    "optimizer_training_summary.csv"
)


optimizer_training_summary.to_csv(
    OPT_TRAINING_SUMMARY_PATH,
    index=False
)


display(
    optimizer_training_summary[
        [
            "optimizer",
            "learning_rate",
            "weight_decay",
            "parameters",
            "best_epoch",
            "best_val_dice_training",
            "val_iou_at_best_dice_epoch",
            "training_minutes"
        ]
    ].round(6)
)


print("\n✓ Saved:")
print(OPT_TRAINING_SUMMARY_PATH)

,optimizer,learning_rate,weight_decay,parameters,best_epoch,best_val_dice_training,val_iou_at_best_dice_epoch,training_minutes
0,Adam,0.0001,0.0000,1940817,30,0.925496,0.863591,10.544691
1,AdamW,0.0001,0.0001,1940817,30,0.925496,0.863591,10.582545



✓ Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/tables/optimizer_training_summary.csv


In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 21
# POSTPROCESSING + VALIDATION METRICS
# ================================================================

def keep_largest_component(mask):

    mask = np.asarray(mask).astype(np.uint8)

    if mask.sum() == 0:
        return np.zeros_like(mask, dtype=bool)

    n_labels, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            mask,
            connectivity=8
        )
    )

    if n_labels <= 1:
        return np.zeros_like(mask, dtype=bool)

    largest_label = (
        1 + np.argmax(
            stats[1:, cv2.CC_STAT_AREA]
        )
    )

    return labels == largest_label


def postprocess_prediction(mask):

    largest = keep_largest_component(mask)

    filled = binary_fill_holes(
        largest
    )

    return np.asarray(
        filled,
        dtype=bool
    )


def binary_dice(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    intersection = np.logical_and(
        gt, pred
    ).sum()

    denominator = gt.sum() + pred.sum()

    if denominator == 0:
        return 1.0

    return (
        2.0 * intersection /
        denominator
    )


def binary_iou(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    intersection = np.logical_and(
        gt, pred
    ).sum()

    union = np.logical_or(
        gt, pred
    ).sum()

    if union == 0:
        return 1.0

    return intersection / union


def binary_precision(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    tp = np.logical_and(gt, pred).sum()
    fp = np.logical_and(~gt, pred).sum()

    if (tp + fp) == 0:
        return 1.0 if gt.sum() == 0 else 0.0

    return tp / (tp + fp)


def binary_recall(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    tp = np.logical_and(gt, pred).sum()
    fn = np.logical_and(gt, ~pred).sum()

    if (tp + fn) == 0:
        return 1.0

    return tp / (tp + fn)


def boundary_points(mask):

    mask = np.asarray(mask).astype(np.uint8)

    kernel = np.ones(
        (3, 3),
        dtype=np.uint8
    )

    eroded = cv2.erode(
        mask,
        kernel,
        iterations=1
    )

    boundary = (
        mask.astype(bool)
        &
        ~eroded.astype(bool)
    )

    return np.column_stack(
        np.where(boundary)
    )


def hd95_pixels(gt, pred):

    gt_points = boundary_points(gt)
    pred_points = boundary_points(pred)

    if (
        len(gt_points) == 0
        or len(pred_points) == 0
    ):
        return np.nan

    distances = cdist(
        gt_points,
        pred_points
    )

    gt_to_pred = distances.min(axis=1)
    pred_to_gt = distances.min(axis=0)

    all_distances = np.concatenate([
        gt_to_pred,
        pred_to_gt
    ])

    return float(
        np.percentile(
            all_distances,
            95
        )
    )


print("✓ Postprocessing and evaluation metrics ready.")

✓ Postprocessing and evaluation metrics ready.


In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 22
# OPTIMIZER CHECKPOINT EVALUATION
# ================================================================

def evaluate_optimizer_checkpoint(
    optimizer_name
):

    run_name = optimizer_name.lower()

    model_path = (
        P1B_MODEL_DIR /
        f"optimizer_{run_name}_best.keras"
    )

    assert model_path.exists(), (
        f"Checkpoint missing:\n{model_path}"
    )

    print("\n" + "=" * 70)
    print(
        f"EVALUATING OPTIMIZER = {optimizer_name}"
    )
    print("=" * 70)

    tf.keras.backend.clear_session()
    gc.collect()

    evaluation_model = (
        tf.keras.models.load_model(
            model_path,
            custom_objects={
                "bce_dice_loss":
                    bce_dice_loss,

                "dice_coefficient":
                    dice_coefficient,

                "iou_metric":
                    iou_metric
            }
        )
    )

    records = []

    for patient_number, pid in enumerate(
        val_ids,
        start=1
    ):

        for phase in ["ED", "ES"]:

            image, gt_mask = load_camus_frame(
                pid,
                phase
            )

            x, gt_resized = preprocess_frame(
                image,
                gt_mask
            )

            probability = evaluation_model(
                x[None, ...],
                training=False
            ).numpy()[0, :, :, 0]

            pred_raw = (
                probability >= THRESHOLD
            )

            pred_mask = (
                postprocess_prediction(
                    pred_raw
                )
            )

            gt_binary = (
                gt_resized[:, :, 0] > 0.5
            )

            gt_area = int(
                gt_binary.sum()
            )

            pred_area = int(
                pred_mask.sum()
            )

            if gt_area > 0:

                signed_area_error_pct = (
                    (pred_area - gt_area)
                    / gt_area
                    * 100.0
                )

                abs_area_error_pct = abs(
                    signed_area_error_pct
                )

            else:

                signed_area_error_pct = np.nan
                abs_area_error_pct = np.nan


            hd95 = hd95_pixels(
                gt_binary,
                pred_mask
            )


            records.append({

                "patient_id":
                    pid,

                "phase":
                    phase,

                "optimizer":
                    optimizer_name,

                "dice":
                    binary_dice(
                        gt_binary,
                        pred_mask
                    ),

                "iou":
                    binary_iou(
                        gt_binary,
                        pred_mask
                    ),

                "precision":
                    binary_precision(
                        gt_binary,
                        pred_mask
                    ),

                "recall":
                    binary_recall(
                        gt_binary,
                        pred_mask
                    ),

                "hd95_pixels":
                    hd95,

                "normalized_hd95":
                    (
                        hd95 /
                        SELECTED_RESOLUTION
                        if np.isfinite(hd95)
                        else np.nan
                    ),

                "gt_area_pixels":
                    gt_area,

                "pred_area_pixels":
                    pred_area,

                "signed_area_error_pct":
                    signed_area_error_pct,

                "abs_area_error_pct":
                    abs_area_error_pct
            })


        if patient_number % 10 == 0:

            print(
                f"Evaluated "
                f"{patient_number}/"
                f"{len(val_ids)} patients"
            )


    results_df = pd.DataFrame(
        records
    )

    output_path = (
        P1B_TABLE_DIR /
        f"optimizer_{run_name}_validation_metrics.csv"
    )

    results_df.to_csv(
        output_path,
        index=False
    )


    print(
        "\nFrames evaluated:",
        len(results_df)
    )

    print("Saved:")
    print(output_path)


    del evaluation_model

    tf.keras.backend.clear_session()
    gc.collect()

    return results_df

In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 23
# EVALUATE BOTH OPTIMIZERS
# ================================================================

adam_eval = (
    evaluate_optimizer_checkpoint(
        "Adam"
    )
)

adamw_eval = (
    evaluate_optimizer_checkpoint(
        "AdamW"
    )
)


assert len(adam_eval) == 100
assert len(adamw_eval) == 100


print("\n✓ Adam evaluated on 100 validation frames.")
print("✓ AdamW evaluated on 100 validation frames.")


EVALUATING OPTIMIZER = Adam
Evaluated 10/50 patients
Evaluated 20/50 patients
Evaluated 30/50 patients
Evaluated 40/50 patients
Evaluated 50/50 patients

Frames evaluated: 100
Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/tables/optimizer_adam_validation_metrics.csv

EVALUATING OPTIMIZER = AdamW
Evaluated 10/50 patients
Evaluated 20/50 patients
Evaluated 30/50 patients
Evaluated 40/50 patients
Evaluated 50/50 patients

Frames evaluated: 100
Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/tables/optimizer_adamw_validation_metrics.csv

✓ Adam evaluated on 100 validation frames.
✓ AdamW evaluated on 100 validation frames.


In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 24
# FINAL OPTIMIZER COMPARISON
# ================================================================

def summarize_optimizer(
    evaluation_df,
    training_result
):

    ed = evaluation_df[
        evaluation_df["phase"] == "ED"
    ]

    es = evaluation_df[
        evaluation_df["phase"] == "ES"
    ]


    return {

        "optimizer":
            training_result[
                "optimizer"
            ],

        "learning_rate":
            training_result[
                "learning_rate"
            ],

        "weight_decay":
            training_result[
                "weight_decay"
            ],

        "parameters":
            training_result[
                "parameters"
            ],

        "best_epoch":
            training_result[
                "best_epoch"
            ],

        "training_minutes":
            training_result[
                "training_minutes"
            ],

        "overall_dice":
            evaluation_df[
                "dice"
            ].mean(),

        "ED_dice":
            ed[
                "dice"
            ].mean(),

        "ES_dice":
            es[
                "dice"
            ].mean(),

        "overall_iou":
            evaluation_df[
                "iou"
            ].mean(),

        "precision":
            evaluation_df[
                "precision"
            ].mean(),

        "recall":
            evaluation_df[
                "recall"
            ].mean(),

        "normalized_hd95":
            evaluation_df[
                "normalized_hd95"
            ].mean(),

        "ED_area_MAE_pct":
            ed[
                "abs_area_error_pct"
            ].mean(),

        "ES_area_MAE_pct":
            es[
                "abs_area_error_pct"
            ].mean(),

        "overall_area_MAE_pct":
            evaluation_df[
                "abs_area_error_pct"
            ].mean()
    }


optimizer_comparison = pd.DataFrame([

    summarize_optimizer(
        adam_eval,
        adam_result
    ),

    summarize_optimizer(
        adamw_eval,
        adamw_result
    )

])


OPT_COMPARISON_PATH = (
    P1B_TABLE_DIR /
    "optimizer_validation_comparison.csv"
)


optimizer_comparison.to_csv(
    OPT_COMPARISON_PATH,
    index=False
)


display(
    optimizer_comparison.round(4)
)


print("\n✓ Final optimizer comparison saved:")
print(OPT_COMPARISON_PATH)

,optimizer,learning_rate,weight_decay,parameters,best_epoch,training_minutes,overall_dice,ED_dice,ES_dice,overall_iou,precision,recall,normalized_hd95,ED_area_MAE_pct,ES_area_MAE_pct,overall_area_MAE_pct
0,Adam,0.0001,0.0000,1940817,30,10.5447,0.9295,0.9401,0.9188,0.8705,0.9312,0.9334,0.0234,6.4293,10.0577,8.2435
1,AdamW,0.0001,0.0001,1940817,30,10.5825,0.9295,0.9401,0.9188,0.8705,0.9312,0.9334,0.0234,6.4293,10.0577,8.2435



✓ Final optimizer comparison saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/tables/optimizer_validation_comparison.csv


In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 25
# ADAM vs ADAMW DIFFERENCE CHECK
# ================================================================

comparison_columns = [
    "dice",
    "iou",
    "precision",
    "recall",
    "normalized_hd95",
    "abs_area_error_pct"
]

differences = {}

for column in comparison_columns:

    difference = np.abs(
        adam_eval[column].to_numpy()
        -
        adamw_eval[column].to_numpy()
    )

    differences[column] = {
        "mean_absolute_difference":
            float(np.nanmean(difference)),

        "maximum_absolute_difference":
            float(np.nanmax(difference))
    }


difference_df = (
    pd.DataFrame(differences)
    .T
    .reset_index()
    .rename(
        columns={
            "index": "metric"
        }
    )
)


display(
    difference_df
)


print(
    "\nIf all differences are exactly or nearly zero, "
    "AdamW produced no practically measurable change "
    "under the predefined weight-decay configuration."
)

,metric,mean_absolute_difference,maximum_absolute_difference
0,dice,0.0,0.0
1,iou,0.0,0.0
2,precision,0.0,0.0
3,recall,0.0,0.0
4,normalized_hd95,0.0,0.0
5,abs_area_error_pct,0.0,0.0



If all differences are exactly or nearly zero, AdamW produced no practically measurable change under the predefined weight-decay configuration.


In [ ]:
# ================================================================
# PHASE 1B-2 — CELL 26
# FINALIZE OPTIMIZER SELECTION
# ================================================================

SELECTED_OPTIMIZER = "Adam"

selected_row = optimizer_comparison[
    optimizer_comparison["optimizer"] == SELECTED_OPTIMIZER
].iloc[0]


optimizer_decision = {

    "experiment":
        "Phase 1B-2 — Controlled Optimizer Experiment",

    "status":
        "COMPLETED",

    "selected_optimizer":
        SELECTED_OPTIMIZER,

    "optimizers_compared":
        ["Adam", "AdamW"],

    "adamw_weight_decay":
        ADAMW_WEIGHT_DECAY,

    "fixed_resolution":
        SELECTED_RESOLUTION,

    "fixed_learning_rate":
        SELECTED_LEARNING_RATE,

    "fixed_architecture":
        "Baseline U-Net",

    "parameter_count":
        BASELINE_PARAMETER_COUNT,

    "fixed_batch_size":
        BATCH_SIZE,

    "fixed_loss":
        "BCE + Dice",

    "augmentation":
        False,

    "epochs":
        MAX_EPOCHS,

    "threshold":
        THRESHOLD,

    "selection_dataset":
        "Validation set only",

    "test_set_used":
        False,

    "selected_validation_results": {

        "overall_dice":
            float(selected_row["overall_dice"]),

        "ED_dice":
            float(selected_row["ED_dice"]),

        "ES_dice":
            float(selected_row["ES_dice"]),

        "overall_iou":
            float(selected_row["overall_iou"]),

        "precision":
            float(selected_row["precision"]),

        "recall":
            float(selected_row["recall"]),

        "normalized_hd95":
            float(selected_row["normalized_hd95"]),

        "ED_area_MAE_pct":
            float(selected_row["ED_area_MAE_pct"]),

        "ES_area_MAE_pct":
            float(selected_row["ES_area_MAE_pct"]),

        "overall_area_MAE_pct":
            float(selected_row["overall_area_MAE_pct"])
    },

    "selection_reason": (
        "Adam and AdamW with weight decay 1e-4 produced identical "
        "training and postprocessed validation results under the "
        "controlled experimental configuration. Per-frame differences "
        "in Dice, IoU, precision, recall, normalized HD95, and absolute "
        "area error were all zero. Since AdamW provided no measurable "
        "validation improvement while introducing an additional "
        "weight-decay hyperparameter, Adam was retained as the simpler "
        "optimizer configuration."
    ),

    "important_interpretation": (
        "This result does not establish that Adam is universally superior "
        "to AdamW. It only shows that the tested AdamW configuration "
        "(weight decay 1e-4) provided no measurable advantage in this "
        "experiment."
    ),

    "next_experiment":
        "Phase 1B-3 — Controlled Batch Size Experiment"
}


OPT_DECISION_PATH = (
    P1B_CONFIG_DIR /
    "phase1B2_optimizer_decision.json"
)


with open(
    OPT_DECISION_PATH,
    "w"
) as f:

    json.dump(
        optimizer_decision,
        f,
        indent=2
    )


print("=" * 70)
print("PHASE 1B-2 COMPLETED")
print("=" * 70)

print("\nSelected optimizer:")
print(SELECTED_OPTIMIZER)

print("\nReason:")
print(
    "Adam and AdamW produced identical validation results; "
    "Adam retained as the simpler configuration."
)

print("\nLOCKED SETTINGS")
print(
    f"Resolution    : "
    f"{SELECTED_RESOLUTION}×{SELECTED_RESOLUTION}"
)
print(
    f"Learning rate : "
    f"{SELECTED_LEARNING_RATE}"
)
print(
    f"Optimizer     : "
    f"{SELECTED_OPTIMIZER}"
)

print("\nValidation performance:")
print(
    f"Overall Dice  : "
    f"{selected_row['overall_dice']:.4f}"
)
print(
    f"ED Dice       : "
    f"{selected_row['ED_dice']:.4f}"
)
print(
    f"ES Dice       : "
    f"{selected_row['ES_dice']:.4f}"
)
print(
    f"Overall IoU   : "
    f"{selected_row['overall_iou']:.4f}"
)
print(
    f"Overall Area MAE: "
    f"{selected_row['overall_area_MAE_pct']:.2f}%"
)

print("\nTest set used: NO")

print("\nNext:")
print("Phase 1B-3 — Controlled Batch Size Experiment")

print("\nDecision saved:")
print(OPT_DECISION_PATH)

print("\n✓ Optimizer officially locked at Adam.")

PHASE 1B-2 COMPLETED

Selected optimizer:
Adam

Reason:
Adam and AdamW produced identical validation results; Adam retained as the simpler configuration.

LOCKED SETTINGS
Resolution    : 256×256
Learning rate : 0.0001
Optimizer     : Adam

Validation performance:
Overall Dice  : 0.9295
ED Dice       : 0.9401
ES Dice       : 0.9188
Overall IoU   : 0.8705
Overall Area MAE: 8.24%

Test set used: NO

Next:
Phase 1B-3 — Controlled Batch Size Experiment

Decision saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/configs/phase1B2_optimizer_decision.json

✓ Optimizer officially locked at Adam.
